In [3]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from dotenv import load_dotenv
import os

# PART 1 -Getting Started with Groq API

## Task 1: Groq API Setup & Basic Chat

In [4]:
load_dotenv()

True

In [5]:
Groq_api = os.getenv('GROQ_API_Key')

####  !pip install groq langchain-groq

In [6]:
llm = ChatGroq(model='llama-3.1-8b-instant')

In [7]:
Response = llm.invoke('Who are you ?')

In [8]:
Response.content

"I'm an artificial intelligence model known as a large language model (LLM) or a conversational AI. I'm a computer program designed to understand and generate human-like text responses. I'm here to assist you with any questions, provide information, or engage in conversation on a wide range of topics.\n\nI don't have a personal identity or emotions like humans do, but I'm designed to be helpful, informative, and engaging. I can understand natural language, recognize context, and respond accordingly. My goal is to provide accurate and useful information, answer your questions, or simply chat with you.\n\nI'm a machine learning model, which means I've been trained on a massive dataset of text from various sources. This training allows me to recognize patterns, understand relationships between words, and generate text that's coherent and contextually relevant.\n\nFeel free to ask me anything, and I'll do my best to help."

## Task 2: Build a Groq Chatbot (Core Logic)

In [13]:
def groq_chat(prompt :str):

    llm = ChatGroq(model='llama-3.1-8b-instant')
    response = llm.invoke(prompt)

    return response.content

In [14]:
prompt = ChatPromptTemplate.from_messages([('system','You are an Helpfulll AI Assistant'),('human','Who are you ?')])

In [15]:
input = 'Who are you ?'

In [16]:
groq_chat(input)

"I'm an artificial intelligence model known as a large language model (LLM) or conversational AI. I'm a computer program designed to understand and generate human-like text responses to a wide range of questions and topics.\n\nI don't have a personal identity, emotions, or consciousness like a human being. I exist solely to provide information, answer questions, and engage in conversation to the best of my knowledge based on my training data.\n\nMy primary goals are to:\n\n1. Provide accurate and helpful information on various topics.\n2. Assist with tasks such as writing, proofreading, and language translation.\n3. Engage in conversations and answer questions to the best of my ability.\n\nI'm constantly learning and improving my responses based on the conversations I have with users like you. I don't have personal opinions or biases, and I strive to provide neutral and informative responses.\n\nHow can I assist you today?"

In [17]:
groq_chat("what is your age ?")

'I was released to the public in 2023.'

# PART 2 - Groq + RAG

## Task 3 : Groq Based RAG Pipeline

In [18]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_openai import OpenAI
from langchain_chroma import Chroma
from langchain_classic.chains.retrieval import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from dotenv import load_dotenv
load_dotenv()

/var/folders/cf/t0bj31ws20944pwkbddw0ylh0000gn/T/ipykernel_816/664612757.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
USER_AGENT environment variable not set, consider setting it to identify your requests.


True

In [19]:
loader = WebBaseLoader('https://www.amazon.in/dp/B0DSKL9MQ8?_encoding=UTF8&ref_=cct_cg_Budget_2b1&pf_rd_p=4c7d86e9-7005-4b46-99de-a03abbb7c819&pf_rd_r=0QVP772NK55ZAJ83DJ98&th=1')

In [20]:
doc = loader.load()

In [21]:
splitter = RecursiveCharacterTextSplitter(chunk_size=2000,chunk_overlap=500)

In [22]:
chunks = splitter.split_documents(doc)

In [23]:
# chunks

In [24]:
len(chunks)

47

In [25]:
embeddings = OpenAIEmbeddings()

In [26]:
vector = Chroma.from_documents(documents=chunks,embedding=embeddings)

In [27]:
retriver = vector.as_retriever()

In [28]:
from langchain_core.prompts import ChatPromptTemplate,PromptTemplate

In [29]:
systemPrompt = '''you are an Helpful Question Answering AI System. 
                Answer the query from the provided context. If you don't know the answer directly say thank you and 
                I don't know the answer. Keep the answer concise. {context}'''

In [30]:
question="What is the price of the Phone ?"

In [31]:
prompt = ChatPromptTemplate([('system',systemPrompt),('human',question)])

In [32]:
from langchain_groq import ChatGroq

In [33]:
llm = ChatGroq(model='llama-3.1-8b-instant')

In [34]:
QAChain = create_stuff_documents_chain(llm,prompt)

In [35]:
rag = create_retrieval_chain(retriver,QAChain)

In [36]:
result = rag.invoke({'input':question})

In [37]:
result

{'input': 'What is the price of the Phone ?',
 'context': [Document(id='288a3e10-e2e8-4a5a-a509-eecb315d8881', metadata={'source': 'https://www.amazon.in/dp/B0DSKL9MQ8?_encoding=UTF8&ref_=cct_cg_Budget_2b1&pf_rd_p=4c7d86e9-7005-4b46-99de-a03abbb7c819&pf_rd_r=0QVP772NK55ZAJ83DJ98&th=1', 'description': 'Samsung Galaxy S25 Ultra 5G AI Smartphone (Titanium Whitesilver, 12GB RAM, 256GB Storage), 200MP Camera, S Pen Included, Long Battery Life : Amazon.in: Electronics', 'title': 'Samsung Galaxy S25 Ultra 5G Mobile with Galaxy AI', 'language': 'en-in'}, page_content='Would you like to  \n tell us about a lower price?\n                     \n \n \n\n\n\n\n\n\n\nSamsung Galaxy S25 Ultra 5G AI Smartphone (Titanium Whitesilver, 12GB RAM, 256GB Storage), 200MP Camera, S Pen Included, Long Battery Life\n\nShare:\n                                    \n\n\n\n\n\n\nFound a lower price? Let us know.\nWhere did you see a lower price?\nFields with an asterisk * are required\n\n\nPrice Availability\n\n\n\

In [38]:
result['answer']

'The price of the Samsung Galaxy S25 Ultra 5G AI Smartphone is ₹71,999.00'

## Task 4 : Prompt Template for Groq RAG